In [ ]:
import trueskill
import numpy as np
from tabulate import tabulate

# --- 1. Setup and Environment Configuration ---

# Configure the TrueSkill environment
# c: Draw probability (tau: uncertainty) is often set higher in team sports.
# If you want to use the default values (c=25/6), you can omit this.
env = trueskill.TrueSkill(
    mu=25.0,        # Default mean skill
    sigma=25.0/3.0, # Default standard deviation of skill
    beta=25.0/6.0,  # "Dynamic factor": how much skill can change
    tau=25.0/300.0, # "Draw probability": how wide the skill margin is for a draw
    draw_probability=0.10 # 10% chance of a draw
)
# Make the calculation process quiet for cleaner output
env.make_as_team = True

# --- 2. Initialize Teams ---

# All teams start with the same initial skill (mu=25, sigma=25/3 ~ 8.33)
# A Rating is represented as a tuple: (mu, sigma)
initial_rating = env.create_rating() 

teams = {
    "Reds": initial_rating,
    "Blues": initial_rating,
    "Greens": initial_rating
}

def display_ratings(teams, title="Current Ratings"):
    """Helper function to format and print the current ratings."""
    data = []
    for name, (mu, sigma) in teams.items():
        # Display the "exposure" (mu - 3*sigma), which is often used for ranking.
        exposure = mu - 3 * sigma
        data.append([
            name,
            f"{mu:.2f}",
            f"{sigma:.2f}",
            f"{exposure:.2f}"
        ])
    
    print(f"\n## {title} ##")
    print(tabulate(data, 
                   headers=["Team", "Mean Skill ($\mu$)", "Uncertainty ($\sigma$)", "Exposure ($\mu - 3\sigma$)"], 
                   tablefmt="github"))

# --- 3. Simulate Matches and Update Skills ---

print("--- Initial Ratings ---")
display_ratings(teams, "Initial Ratings")

# A match consists of (list of team ratings), [ranks]
# Ranks are integers, 1 = winner, 2 = 2nd place, etc.

# Match 1: Reds beat Blues
print("\n--- Match 1: Reds (1) vs Blues (2) ---")
# The update function takes two lists: one for the ratings and one for the ranks.
ratings = [[teams["Reds"]], [teams["Blues"]]] # Team ratings are passed as a list of teams
ranks = [1, 2] # 1st place (Reds) vs 2nd place (Blues)
new_ratings = env.rate(ratings, ranks=ranks) 

teams["Reds"] = new_ratings[0][0]
teams["Blues"] = new_ratings[1][0]
display_ratings(teams, "Ratings After Match 1")


# Match 2: Greens draw with Reds
print("\n--- Match 2: Greens (1) vs Reds (1) ---")
# A draw is indicated by giving both teams the same rank.
ratings = [[teams["Greens"]], [teams["Reds"]]]
ranks = [1, 1] # 1st place tie (Draw)
new_ratings = env.rate(ratings, ranks=ranks)

teams["Greens"] = new_ratings[0][0]
teams["Reds"] = new_ratings[1][0]
display_ratings(teams, "Ratings After Match 2")


# Match 3: Blues easily beat Greens
print("\n--- Match 3: Blues (1) vs Greens (2) ---")
ratings = [[teams["Blues"]], [teams["Greens"]]]
ranks = [1, 2]
new_ratings = env.rate(ratings, ranks=ranks)

teams["Blues"] = new_ratings[0][0]
teams["Greens"] = new_ratings[1][0]
display_ratings(teams, "Ratings After Match 3")


# --- 4. Prediction Example ---

def predict_win_probability(env, team1_rating, team2_rating):
    """Calculates the probability that Team 1 beats Team 2."""
    # The `quality` function is used to predict the draw probability, 
    # but the log-odds can be calculated directly.
    
    # Unpack mean and variance
    mu1, sigma1 = team1_rating
    mu2, sigma2 = team2_rating
    
    # Calculate the standardization factor (denominator)
    denominator = np.sqrt(2 * (env.beta**2) + sigma1**2 + sigma2**2)
    
    # Z-score for Team 1 winning
    z_score = (mu1 - mu2) / denominator
    
    # Probability is the CDF of the Normal distribution at the Z-score
    return trueskill.backends.choose_backend().cdf(z_score)


print("\n--- Prediction ---")
prob_reds_vs_greens = predict_win_probability(env, teams["Reds"], teams["Greens"])

print(f"Probability Reds beat Greens: {prob_reds_vs_greens:.2f}")